In [ ]:
import numpy as np
import pandas as pd
import cv2 as cv
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from IPython.display import clear_output
from torch.optim.lr_scheduler import StepLR
from models.generator import ColorGenerator
from models.discriminator import Discriminator
from preprocessing import normalizeLAB, denormalizeLAB

In [ ]:
device = "cuda"

# Import Data

In [ ]:
imgs = np.load("dataset/labs.npy")
masks = np.load("dataset/masks.npy")

In [ ]:
class ImagesDataset(Dataset):
    def __init__(self, imgs, masks):
        assert(imgs.shape[0] == masks.shape[0])
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        return self.imgs[idx], self.masks[idx]

In [ ]:
train_num = int(len(imgs) * 0.8)

train = ImagesDataset(imgs[:train_num], masks[:train_num])
test = ImagesDataset(imgs[train_num:], masks[train_num:])

In [ ]:
train_loader = DataLoader(train, batch_size=128,shuffle=True)

In [ ]:
test_image, test_mask = train[100]

real = cv.cvtColor(denormalizeLAB(test_image), code=cv.COLOR_LAB2RGB)
mask = cv.cvtColor(denormalizeLAB(test_mask), code=cv.COLOR_LAB2RGB)

plt.subplot(1,2,1).imshow(real)
plt.subplot(1,2,2).imshow(mask)

# Build model

In [ ]:
lr = 1e-3
epochs = 50

n_cell = 4
generator = ColorGenerator(output_chanels=2,n_cell=n_cell).to(device)
gen_optimizer = AdamW(params=generator.parameters(),lr=lr)

discriminator = Discriminator().to(device)
dis_optimizer = AdamW(params=discriminator.parameters(),lr=lr)
criterion = nn.BCEWithLogitsLoss()
gen_criterion = nn.L1Loss()

scheduler = StepLR(gen_optimizer, step_size=50, gamma=0.1)

In [ ]:
def show_samples(generator, img, c, epoch):

    show_results(generator, img, c)

    plt.suptitle(f"Epoch = {epoch}")
    plt.savefig(f"CNN-model/train-results/{epoch}.png")
    plt.show()

def show_results(generator, img, c):
    if type(img) == np.ndarray: img = torch.tensor(img)
    if type(c) == np.ndarray: c = torch.tensor(c)

    img = img.to(torch.float32).to(device)
    c = c.to(torch.float32).to(device)

    img = torch.moveaxis(img,-1,-3)
    c = torch.moveaxis(c,-1,-3)

    x = img[:,:1,...]
    y = img[:,1:,...]
    x_hat = generator(x,c)
    x_hat = torch.concat([x,x_hat],dim=1).detach().cpu().numpy()
    x_hat = np.moveaxis(x_hat,-3,-1)
    x_hat = denormalizeLAB(x_hat)

    img = img.clone().detach().cpu().numpy()
    img = np.moveaxis(img,-3,-1)
    img = denormalizeLAB(img)

    n = img.shape[0]

    x = x.detach().cpu().numpy()
    fig, axs = plt.subplots(3,n,figsize=(n,4))

    for i in range(n):
        colored = cv.cvtColor(x_hat[i],code=cv.COLOR_LAB2RGB)
        real = cv.cvtColor(img[i],code=cv.COLOR_LAB2RGB)
        axs[0][i].imshow(real)
        axs[1][i].imshow(x[i][0],cmap="grey")
        axs[2][i].imshow(colored)

        for ax in axs.flat: ax.axis("off")
    
    axs[0][0].set_title("Real color")
    axs[1][0].set_title("Grayscale input")
    axs[2][0].set_title("Generated color")

# Train model

In [ ]:
sample_x, sample_c = test[[1,30,100,-100,-30,-1]]

In [ ]:
generator_loss = []
discriminator_loss = []

for epoch in range(epochs):


    epoch_gen_loss = .0
    epoch_dis_loss = .0
    for idx, batch in enumerate(train_loader):
        img, c = batch
        img = torch.moveaxis(img,-1,-3).to(torch.float32).to(device)
        
        x = img[:,:1,...]
        y = img[:,1:,...]
        
        c = torch.moveaxis(c,-1,-3).to(torch.float32).to(device)

        #train discriminator on correct images
        real_label = torch.ones([img.shape[0], 1], device=device, dtype=torch.float32, requires_grad=True)
        real_pred = discriminator(img)
        real_loss = criterion(real_pred,real_label)
        
        # generate fakes
        generated = generator(x,c)
        fakes = torch.concat([x,generated.detach()],dim=1)

        #train discriminator on fake images
        fake_label = torch.zeros([fakes.shape[0], 1], device=device, dtype=torch.float32, requires_grad=True)
        fake_pred = discriminator(fakes)
        fake_loss = criterion(fake_pred,fake_label)

        # back propagation in discriminator
        dis_sum_loss = fake_loss + real_loss

        dis_optimizer.zero_grad()
        dis_sum_loss.backward()
        dis_optimizer.step()

        epoch_dis_loss = epoch_dis_loss + dis_sum_loss.item()

        #train generator
        generated = generator(x,c)
        fakes = torch.concat([x,generated],dim=1)
        fake_pred = discriminator(fakes)
        fake_label = torch.ones([fakes.shape[0], 1], device=device, dtype=torch.float32, requires_grad=True)
        gen_loss = criterion(fake_pred,fake_label) + criterion(generated,y)
        gen_optimizer.zero_grad()
        gen_loss.backward()
        gen_optimizer.step()

        epoch_gen_loss = epoch_gen_loss + gen_loss.item()
    
    clear_output()
    print(f"epoch {epoch} Generator loss: {epoch_gen_loss} Discriminator loss: {epoch_dis_loss}")
    show_samples(generator, sample_x, sample_c,epoch + 1)

    generator_loss.append(epoch_gen_loss)
    discriminator_loss.append(epoch_dis_loss)

    scheduler.step()
    

In [ ]:
rand = torch.rand_like(fakes)

(discriminator(rand)==1).all()

# Test with custom colors